# Gold Fact — Daily Consumption

Builds the Gold daily-consumption fact table from validated Silver smart-meter readings.

## Grain

**One row per Household × Date**

## Purpose

- Aggregate half-hourly readings to household-day level
- Calculate daily consumption and consumption statistics
- Measure daily reading completeness
- Resolve Gold dimension surrogate keys
- Validate referential integrity and business-key uniqueness
- Reconcile Gold consumption totals back to Silver

> This fact table provides the primary analytical grain for household consumption, trends, completeness and benchmark analysis in the semantic model.

## 1. Load Silver Data and Gold Dimensions

Load the validated Silver readings and the conformed dimensions required to construct the daily-consumption fact.

In [1]:
from pyspark.sql import functions as F

silver_df = spark.table("silver.meter_readings")

dim_household_df = spark.table("gold.dim_household")
dim_date_df = spark.table("gold.dim_date")

print("Daily consumption fact build initialised.")

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 3, Finished, Available, Finished, False)

Daily consumption fact build initialised.


## 2. Aggregate to Household-Day Grain

Aggregate validated half-hourly readings to one record per household and date.

For each household-day, calculate:

- Total daily consumption
- Average half-hourly consumption
- Peak half-hourly consumption
- Minimum half-hourly consumption
- Number of readings observed

In [2]:
daily_agg_df = (
    silver_df
    .groupBy(
        "HouseholdID",
        "ReadingDate"
    )
    .agg(
        F.sum("ConsumptionKWh").alias("DailyConsumptionKWh"),
        F.avg("ConsumptionKWh").alias("AverageHalfHourlyKWh"),
        F.max("ConsumptionKWh").alias("PeakHalfHourlyKWh"),
        F.min("ConsumptionKWh").alias("MinimumHalfHourlyKWh"),
        F.count("*").alias("ReadingCount")
    )
)

print(f"Daily aggregated rows: {daily_agg_df.count():,}")

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 4, Finished, Available, Finished, False)

Daily aggregated rows: 3,510,403


## 3. Classify Daily Completeness

A complete household-day is expected to contain **48 half-hourly readings**.

`IsCompleteDay` identifies whether the observed `ReadingCount` equals the expected reading count.

Partial household-days are retained in the fact table rather than removed, allowing completeness to remain visible and measurable in downstream reporting.

In [3]:
daily_agg_df = (
    daily_agg_df
    .withColumn(
        "ExpectedReadingCount",
        F.lit(48)
    )
    .withColumn(
        "IsCompleteDay",
        F.col("ReadingCount") == 48
    )
)

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 5, Finished, Available, Finished, False)

## 4. Resolve Dimension Keys

Join the daily aggregation to the conformed Gold dimensions and replace descriptive business attributes with analytical surrogate keys.

The resulting fact references:

- `HouseholdKey`
- `DateKey`
- `TariffKey`

In [4]:
daily_with_household_df = (
    daily_agg_df
    .join(
        dim_household_df.select(
            "HouseholdKey",
            "HouseholdID",
            "TariffKey"
        ),
        on="HouseholdID",
        how="left"
    )
)

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 6, Finished, Available, Finished, False)

In [5]:
daily_with_keys_df = (
    daily_with_household_df
    .join(
        dim_date_df.select(
            "DateKey",
            "Date"
        ),
        daily_with_household_df["ReadingDate"] == dim_date_df["Date"],
        how="left"
    )
)

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 7, Finished, Available, Finished, False)

## 5. Prepare Daily Fact Schema

The final daily-consumption fact contains:

- `HouseholdKey`
- `DateKey`
- `TariffKey`
- `DailyConsumptionKWh`
- `AverageHalfHourlyKWh`
- `PeakHalfHourlyKWh`
- `MinimumHalfHourlyKWh`
- `ReadingCount`
- `ExpectedReadingCount`
- `IsCompleteDay`

The table contains no household or calendar descriptions because those attributes are supplied by the related dimensions.

In [6]:
fact_daily_df = (
    daily_with_keys_df
    .select(
        "HouseholdKey",
        "DateKey",
        "TariffKey",
        "DailyConsumptionKWh",
        "AverageHalfHourlyKWh",
        "PeakHalfHourlyKWh",
        "MinimumHalfHourlyKWh",
        "ReadingCount",
        "ExpectedReadingCount",
        "IsCompleteDay"
    )
)

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 8, Finished, Available, Finished, False)

In [7]:
key_validation_df = (
    fact_daily_df
    .agg(
        F.sum(
            F.when(F.col("HouseholdKey").isNull(), 1)
            .otherwise(0)
        ).alias("MissingHouseholdKeys"),

        F.sum(
            F.when(F.col("DateKey").isNull(), 1)
            .otherwise(0)
        ).alias("MissingDateKeys"),

        F.sum(
            F.when(F.col("TariffKey").isNull(), 1)
            .otherwise(0)
        ).alias("MissingTariffKeys")
    )
)

display(key_validation_df)

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 53e30391-2a7e-4c8f-a3cf-c2762452499f)

In [8]:
duplicate_fact_keys = (
    fact_daily_df
    .groupBy(
        "HouseholdKey",
        "DateKey"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate fact business keys: {duplicate_fact_keys:,}")

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 10, Finished, Available, Finished, False)

Duplicate fact business keys: 0


In [9]:
print("DAILY FACT COMPLETENESS")
print("-" * 50)

fact_daily_df.groupBy(
    "IsCompleteDay"
).count().show()

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 11, Finished, Available, Finished, False)

DAILY FACT COMPLETENESS
--------------------------------------------------
+-------------+-------+
|IsCompleteDay|  count|
+-------------+-------+
|         true|3469352|
|        false|  41051|
+-------------+-------+



In [10]:
silver_consumption = (
    silver_df
    .agg(
        F.sum("ConsumptionKWh").alias("TotalConsumption")
    )
    .collect()[0]["TotalConsumption"]
)

gold_consumption = (
    fact_daily_df
    .agg(
        F.sum("DailyConsumptionKWh").alias("TotalConsumption")
    )
    .collect()[0]["TotalConsumption"]
)

difference = silver_consumption - gold_consumption

print("CONSUMPTION RECONCILIATION")
print("-" * 50)
print(f"Silver consumption: {silver_consumption:,.6f}")
print(f"Gold consumption:   {gold_consumption:,.6f}")
print(f"Difference:         {difference:,.12f}")

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 12, Finished, Available, Finished, False)

CONSUMPTION RECONCILIATION
--------------------------------------------------
Silver consumption: 35,539,823.306363
Gold consumption:   35,539,823.306385
Difference:         -0.000021681190


## 6. Persist Daily Consumption Fact

Persist the household-day aggregation as the Delta-backed Gold table:

`gold.fact_daily_consumption`

In [11]:
(
    fact_daily_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.fact_daily_consumption")
)

print("gold.fact_daily_consumption written successfully.")

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 13, Finished, Available, Finished, False)

gold.fact_daily_consumption written successfully.


## 7. Validate Fact Integrity

Validate the persisted fact table before exposing it to downstream analytics.

Quality checks include:

- Fact row count
- Missing household keys
- Missing date keys
- Missing tariff keys
- Duplicate Household × Date business keys
- Daily completeness distribution

In [12]:
persisted_daily_df = spark.table(
    "gold.fact_daily_consumption"
)

print("FACT DAILY CONSUMPTION")
print("-" * 50)
print(f"Rows: {persisted_daily_df.count():,}")

persisted_daily_df.printSchema()

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 14, Finished, Available, Finished, False)

FACT DAILY CONSUMPTION
--------------------------------------------------
Rows: 3,510,403
root
 |-- HouseholdKey: integer (nullable = true)
 |-- DateKey: integer (nullable = true)
 |-- TariffKey: integer (nullable = true)
 |-- DailyConsumptionKWh: double (nullable = true)
 |-- AverageHalfHourlyKWh: double (nullable = true)
 |-- PeakHalfHourlyKWh: double (nullable = true)
 |-- MinimumHalfHourlyKWh: double (nullable = true)
 |-- ReadingCount: long (nullable = true)
 |-- ExpectedReadingCount: integer (nullable = true)
 |-- IsCompleteDay: boolean (nullable = true)



## 9. Inspect Gold Fact Storage

Inspect the physical Delta representation of the aggregated fact to understand the reduction achieved by moving from half-hourly Silver readings to household-day analytical grain.

In [13]:
spark.sql(
    "DESCRIBE DETAIL gold.fact_daily_consumption"
).select(
    "numFiles",
    "sizeInBytes",
    "partitionColumns"
).show(truncate=False)

StatementMeta(, 76406f19-36cf-4b50-9512-cac22a3a2196, 15, Finished, Available, Finished, False)

+--------+-----------+----------------+
|numFiles|sizeInBytes|partitionColumns|
+--------+-----------+----------------+
|19      |54336474   |[]              |
+--------+-----------+----------------+



## Daily Fact Summary

The Gold daily-consumption fact successfully converts the validated reading-level dataset into a household-day analytical model.

- **3,510,403** household-day records
- **3,469,352** complete household-days
- **41,051** partial household-days
- **0** missing dimension keys
- **0** duplicate Household × Date keys
- **35.54M kWh** of consumption reconciled to Silver with only negligible floating-point variance

The resulting fact supports daily consumption trends, household analysis, completeness monitoring and portfolio-level benchmarking in the Direct Lake semantic model.